**Check whether GPU is connected**

In [ ]:
!nvidia-smi

Tue Nov 17 15:27:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import sys

print("Python version:", sys.version)

try:
    import numba
    print("Numba version:", numba.__version__)
except ImportError:
    print("Numba is not installed")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Numba version: 0.64.0


**Check whether Google Colab is connected to an NVIDIA Tesla T4 GPU**

In [ ]:
from numba import cuda

print("CUDA Available:", cuda.is_available())
print("GPU Count:", len(cuda.gpus))

if cuda.is_available():
    print("GPU Details:")
    print(cuda.get_current_device())

CUDA Available: True
GPU Count: 1
GPU Details:
<CUDA device 0 'Tesla T4'>


**First CUDA program**

In [ ]:

import numpy as np
from numba import cuda

# GPU function (runs on the CUDA GPU)
@cuda.jit
def add_arrays(a, b, result):
    i = cuda.grid(1)

    if i < result.size:
        result[i] = a[i] + b[i]

# Create two arrays on the CPU
a = np.array([10, 20, 30, 40, 50], dtype=np.float32)
b = np.array([1, 2, 3, 4, 5], dtype=np.float32)

# Copy arrays to GPU memory
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_result = cuda.device_array(a.size, dtype=np.float32)

# Run the CUDA function on the GPU
threads_per_block = 128
blocks_per_grid = 1

add_arrays[blocks_per_grid, threads_per_block](
    d_a, d_b, d_result
)

# Copy result back from GPU to CPU
result = d_result.copy_to_host()

print("Array A:", a)
print("Array B:", b)
print("GPU Result:", result)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Array A: [10. 20. 30. 40. 50.]
Array B: [1. 2. 3. 4. 5.]
GPU Result: [11. 22. 33. 44. 55.]


**Generate the dataset using Python**

In [ ]:

import numpy as np

# Generate 1 million random numbers
dataset = np.random.randint(
    1, 1001,
    size=1_000_000
).astype(np.float32)

# Display dataset information
print("Dataset created successfully!")
print("Number of values:", len(dataset))
print("Data type:", dataset.dtype)
print("First 10 values:", dataset[:10])
print("Minimum value:", np.min(dataset))
print("Maximum value:", np.max(dataset))


Dataset created successfully!
Number of values: 1000000
Data type: float32
First 10 values: [771. 152. 529. 795. 485. 474. 625. 437. 209. 505.]
Minimum value: 1.0
Maximum value: 1000.0


**Calculate sum using CPU**

In [ ]:
import time
import numpy as np

# Start CPU timer
cpu_start = time.perf_counter()

# Calculate sum using CPU
cpu_sum = np.sum(dataset)

# Stop CPU timer
cpu_end = time.perf_counter()

# Calculate execution time
cpu_time = cpu_end - cpu_start

print("CPU SUM RESULT")
print("----------------------")
print("Total Sum:", cpu_sum)
print("Execution Time:", cpu_time, "seconds")


CPU SUM RESULT
----------------------
Total Sum: 4.999354e+08
Execution Time: 0.0006776320000199121 seconds


**Calculate sum using CUDA (GPU)**

In [ ]:

import time
import numpy as np
from numba import cuda, float32

# First GPU kernel: calculate partial sums
@cuda.jit
def partial_sum(data, partials, n):
    tid = cuda.threadIdx.x
    idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    # Each thread adds multiple values
    total = float32(0.0)

    while idx < n:
        total += data[idx]
        idx += stride

    # Store each thread's sum in shared memory
    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = total

    cuda.syncthreads()

    # Combine thread sums within each block
    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            shared[tid] += shared[tid + step]

        cuda.syncthreads()
        step //= 2

    # One partial sum per block
    if tid == 0:
        partials[cuda.blockIdx.x] = shared[0]


# Second GPU kernel: calculate the final sum
@cuda.jit
def final_sum(partials, result, n):
    tid = cuda.threadIdx.x

    total = float32(0.0)

    # Each thread adds some partial sums
    for i in range(tid, n, cuda.blockDim.x):
        total += partials[i]

    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = total

    cuda.syncthreads()

    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            shared[tid] += shared[tid + step]

        cuda.syncthreads()
        step //= 2

    if tid == 0:
        result[0] = shared[0]


# Copy dataset from CPU to GPU
d_dataset = cuda.to_device(dataset)

# Configure GPU execution
threads = 256
blocks = 256

d_partials = cuda.device_array(blocks, dtype=np.float32)
d_result = cuda.device_array(1, dtype=np.float32)

# Warm-up: compile and execute kernels once
partial_sum[blocks, threads](d_dataset, d_partials, len(dataset))
final_sum[1, threads](d_partials, d_result, blocks)
cuda.synchronize()

# Measure GPU kernel execution time
gpu_start = time.perf_counter()

partial_sum[blocks, threads](d_dataset, d_partials, len(dataset))
final_sum[1, threads](d_partials, d_result, blocks)

cuda.synchronize()
gpu_end = time.perf_counter()

# Copy final result back to CPU
gpu_sum = d_result.copy_to_host()[0]

gpu_time = gpu_end - gpu_start

# Display results
print("GPU SUM RESULT")
print("----------------------")
print("GPU Sum:", gpu_sum)
print("GPU Execution Time:", gpu_time, "seconds")

print("\nCORRECTNESS CHECK")
print("----------------------")
print("CPU Sum:", cpu_sum)
print("Results Match:", np.isclose(cpu_sum, gpu_sum, rtol=1e-5))


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU SUM RESULT
----------------------
GPU Sum: 4.9993542e+08
GPU Execution Time: 0.0007122849999916525 seconds

CORRECTNESS CHECK
----------------------
CPU Sum: 4.999354e+08
Results Match: True


first CUDA parallel reduction is working correctly. We have successfully calculated the sum of one million numbers using both CPU and GPU.
Correctness check passed
1. Understand your results
Measurement	CPU	GPU
Sum	499,935,400	499,935,420 approximately
Execution time	0.000678 s	0.000712 s
Result	Correct	Correct

The tiny difference in the displayed sum is due to floating-point rounding.
Why is the GPU slightly slower?
This is normal for our current experiment.
- NumPy's CPU sum is highly optimized.
- Our GPU calculation launches two CUDA kernels.
- For one million numbers, the GPU's launch overhead can reduce its advantage.
- We haven't included data-transfer time in the GPU measurement.
Don't worry about this result. We'll test larger datasets and optimize our implementation before drawing conclusions.

The warning about grid size 1 comes from the final reduction kernel, which uses one block to combine the 256 partial sums. It is expected for this design.

**Calculate average using CPU and GPU**

In [ ]:

# Calculate average using CPU
cpu_average = cpu_sum / len(dataset)

# Calculate average using GPU result
gpu_average = gpu_sum / len(dataset)

print("AVERAGE RESULTS")
print("----------------------")

print("CPU Average:", cpu_average)
print("GPU Average:", gpu_average)

print("\nCORRECTNESS CHECK")
print("----------------------")

print(
    "Results Match:",
    np.isclose(cpu_average, gpu_average, rtol=1e-5)
)


AVERAGE RESULTS
----------------------
CPU Average: 499.9354
GPU Average: 499.93542

CORRECTNESS CHECK
----------------------
Results Match: True


**Find the minimum value using CUDA**

In [ ]:

# Find minimum value using GPU
gpu_min = cuda.to_device(np.array([np.inf], dtype=np.float32))

@cuda.jit
def find_min(data, partials, n):
    tid = cuda.threadIdx.x
    idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    local_min = float32(np.inf)

    while idx < n:
        if data[idx] < local_min:
            local_min = data[idx]
        idx += stride

    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = local_min
    cuda.syncthreads()

    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            if shared[tid + step] < shared[tid]:
                shared[tid] = shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        partials[cuda.blockIdx.x] = shared[0]


@cuda.jit
def final_min(partials, result, n):
    tid = cuda.threadIdx.x
    local_min = float32(np.inf)

    for i in range(tid, n, cuda.blockDim.x):
        if partials[i] < local_min:
            local_min = partials[i]

    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = local_min
    cuda.syncthreads()

    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            if shared[tid + step] < shared[tid]:
                shared[tid] = shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        result[0] = shared[0]


# Execute on GPU
blocks = 256
threads = 256

d_min_partials = cuda.device_array(blocks, dtype=np.float32)
d_min_result = cuda.device_array(1, dtype=np.float32)

find_min[blocks, threads](d_dataset, d_min_partials, len(dataset))
final_min[1, threads](d_min_partials, d_min_result, blocks)
cuda.synchronize()

gpu_min = d_min_result.copy_to_host()[0]
cpu_min = np.min(dataset)

print("CPU Minimum:", cpu_min)
print("GPU Minimum:", gpu_min)
print("Results Match:", np.isclose(cpu_min, gpu_min))


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


CPU Minimum: 1.0
GPU Minimum: 1.0
Results Match: True


**Find the maximum value using CUDA**

In [ ]:

@cuda.jit
def find_max(data, partials, n):
    tid = cuda.threadIdx.x
    idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    local_max = float32(-np.inf)

    while idx < n:
        if data[idx] > local_max:
            local_max = data[idx]
        idx += stride

    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = local_max
    cuda.syncthreads()

    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            if shared[tid + step] > shared[tid]:
                shared[tid] = shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        partials[cuda.blockIdx.x] = shared[0]


@cuda.jit
def final_max(partials, result, n):
    tid = cuda.threadIdx.x
    local_max = float32(-np.inf)

    for i in range(tid, n, cuda.blockDim.x):
        if partials[i] > local_max:
            local_max = partials[i]

    shared = cuda.shared.array(256, dtype=float32)
    shared[tid] = local_max
    cuda.syncthreads()

    step = cuda.blockDim.x // 2

    while step > 0:
        if tid < step:
            if shared[tid + step] > shared[tid]:
                shared[tid] = shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        result[0] = shared[0]


# Execute on GPU
d_max_partials = cuda.device_array(blocks, dtype=np.float32)
d_max_result = cuda.device_array(1, dtype=np.float32)

find_max[blocks, threads](d_dataset, d_max_partials, len(dataset))
final_max[1, threads](d_max_partials, d_max_result, blocks)
cuda.synchronize()

gpu_max = d_max_result.copy_to_host()[0]
cpu_max = np.max(dataset)

print("CPU Maximum:", cpu_max)
print("GPU Maximum:", gpu_max)
print("Results Match:", np.isclose(cpu_max, gpu_max))


CPU Maximum: 1000.0
GPU Maximum: 1000.0
Results Match: True


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


**Test with a larger dataset (10 million numbers)**

In [ ]:

# Generate 10 million random numbers
large_dataset = np.random.randint(
    1, 1001, size=10_000_000
).astype(np.float32)

print("Large dataset created!")
print("Number of values:", len(large_dataset))
print("Memory size:", large_dataset.nbytes / (1024 * 1024), "MB")


Large dataset created!
Number of values: 10000000
Memory size: 38.14697265625 MB


**Calculate sum using CPU (10 million numbers)**

In [ ]:

# CPU sum and execution time
start = time.perf_counter()

large_cpu_sum = np.sum(large_dataset)

cpu_large_time = time.perf_counter() - start

print("CPU SUM RESULT")
print("----------------------")
print("CPU Sum:", large_cpu_sum)
print("CPU Execution Time:", cpu_large_time, "seconds")


CPU SUM RESULT
----------------------
CPU Sum: 5.004917e+09
CPU Execution Time: 0.010353616999964288 seconds


**Calculate sum using GPU**

In [ ]:

# Copy the 10-million dataset to GPU
d_large_dataset = cuda.to_device(large_dataset)

# Configure GPU
blocks = 256
threads = 256

d_large_partials = cuda.device_array(blocks, dtype=np.float32)
d_large_result = cuda.device_array(1, dtype=np.float32)

# Warm-up
partial_sum[blocks, threads](
    d_large_dataset, d_large_partials, len(large_dataset)
)
final_sum[1, threads](d_large_partials, d_large_result, blocks)
cuda.synchronize()

# Measure GPU execution time
start = time.perf_counter()

partial_sum[blocks, threads](
    d_large_dataset, d_large_partials, len(large_dataset)
)
final_sum[1, threads](d_large_partials, d_large_result, blocks)

cuda.synchronize()
gpu_large_time = time.perf_counter() - start

# Get final result
large_gpu_sum = d_large_result.copy_to_host()[0]

print("GPU SUM RESULT")
print("----------------------")
print("GPU Sum:", large_gpu_sum)
print("GPU Execution Time:", gpu_large_time, "seconds")

print("\nCORRECTNESS CHECK")
print("----------------------")
print("CPU Sum:", large_cpu_sum)
print("Results Match:", np.isclose(
    large_cpu_sum, large_gpu_sum, rtol=1e-5
))


GPU SUM RESULT
----------------------
GPU Sum: 5.0049167e+09
GPU Execution Time: 0.005475004999993871 seconds

CORRECTNESS CHECK
----------------------
CPU Sum: 5.004917e+09
Results Match: True


Compare CPU and GPU performance


Your results:
Measurement	CPU	GPU
Sum	5.004917 × 10⁹	5.0049167 × 10⁹
Execution time	0.01035 s	0.00548 s
Correctness	Passed	Passed

GPU speedup
1.89×
GPU kernel execution was approximately 1.89 times faster.

**calculate speedup**

In [ ]:

speedup = cpu_large_time / gpu_large_time

print("CPU Time:", cpu_large_time)
print("GPU Time:", gpu_large_time)
print("GPU Speedup:", round(speedup, 2), "x")


CPU Time: 0.010353616999964288
GPU Time: 0.005475004999993871
GPU Speedup: 1.89 x


**Measure GPU data-transfer time**

In [ ]:

# Measure CPU-to-GPU data transfer
start = time.perf_counter()

d_test = cuda.to_device(large_dataset)

cuda.synchronize()

transfer_time = time.perf_counter() - start

print("DATA TRANSFER RESULT")
print("----------------------")
print("Dataset size:", len(large_dataset))
print("Transfer time:", transfer_time, "seconds")


DATA TRANSFER RESULT
----------------------
Dataset size: 10000000
Transfer time: 0.01099519599983978 seconds


Why are we checking this?

Our earlier GPU speedup excluded data-transfer time. In real applications, data must often be copied to the GPU before processing.

We'll use this measurement to understand the actual cost of GPU processing.

**Calculate total GPU time**

In [ ]:

total_gpu_time = transfer_time + gpu_large_time

real_speedup = cpu_large_time / total_gpu_time

print("PERFORMANCE COMPARISON")
print("----------------------")
print("CPU Time:", cpu_large_time)
print("GPU Kernel Time:", gpu_large_time)
print("Data Transfer Time:", transfer_time)
print("Total GPU Time:", total_gpu_time)
print("Overall Speedup:", round(real_speedup, 2), "x")


PERFORMANCE COMPARISON
----------------------
CPU Time: 0.010353616999964288
GPU Kernel Time: 0.005475004999993871
Data Transfer Time: 0.01099519599983978
Total GPU Time: 0.01647020099983365
Overall Speedup: 0.63 x


What did we discover?

Measurement	Time

CPU calculation	0.01035 s

GPU calculation	0.00548 s

Data transfer	0.01100 s

Total GPU time	0.01647 s

0.63×

The GPU kernel itself is faster, but copying the dataset to the GPU takes additional time. So, for this experiment, CPU execution is faster overall.

This is a valuable project result: GPU acceleration is not automatically faster for every workload. Data-transfer overhead matters.

**Test the 50-million dataset**

In [ ]:

# Generate 50 million random numbers
very_large_dataset = np.random.randint(
    1, 1001, size=50_000_000
).astype(np.float32)

print("Very large dataset created!")
print("Number of values:", len(very_large_dataset))
print("Memory size:", very_large_dataset.nbytes / (1024 * 1024), "MB")


Very large dataset created!
Number of values: 50000000
Memory size: 190.73486328125 MB


**Calculate sum using CPU**

In [ ]:

start = time.perf_counter()

very_large_cpu_sum = np.sum(very_large_dataset)

cpu_very_large_time = time.perf_counter() - start

print("CPU SUM RESULT")
print("----------------------")
print("CPU Sum:", very_large_cpu_sum)
print("CPU Execution Time:", cpu_very_large_time, "seconds")


CPU SUM RESULT
----------------------
CPU Sum: 2.5025372e+10
CPU Execution Time: 0.02285477100008393 seconds


**Calculate sum using GPU**

In [ ]:

# Copy dataset to GPU
d_very_large = cuda.to_device(very_large_dataset)

blocks = 256
threads = 256

d_very_large_partials = cuda.device_array(blocks, dtype=np.float32)
d_very_large_result = cuda.device_array(1, dtype=np.float32)

# Warm-up
partial_sum[blocks, threads](
    d_very_large, d_very_large_partials,
    len(very_large_dataset)
)
final_sum[1, threads](
    d_very_large_partials, d_very_large_result, blocks
)
cuda.synchronize()

# Measure GPU kernel execution
start = time.perf_counter()

partial_sum[blocks, threads](
    d_very_large, d_very_large_partials,
    len(very_large_dataset)
)
final_sum[1, threads](
    d_very_large_partials, d_very_large_result, blocks
)

cuda.synchronize()
gpu_very_large_time = time.perf_counter() - start

very_large_gpu_sum = d_very_large_result.copy_to_host()[0]

print("GPU SUM RESULT")
print("----------------------")
print("GPU Sum:", very_large_gpu_sum)
print("GPU Execution Time:", gpu_very_large_time, "seconds")

print("\nCORRECTNESS CHECK")
print("----------------------")
print("CPU Sum:", very_large_cpu_sum)
print("Results Match:", np.isclose(
    very_large_cpu_sum, very_large_gpu_sum, rtol=1e-5
))


GPU SUM RESULT
----------------------
GPU Sum: 2.5025372e+10
GPU Execution Time: 0.0011556350000319071 seconds

CORRECTNESS CHECK
----------------------
CPU Sum: 2.5025372e+10
Results Match: True


**Compare CPU and GPU performance**

In [ ]:

# Calculate GPU speedup
speedup_50m = cpu_very_large_time / gpu_very_large_time

print("50 MILLION DATASET PERFORMANCE")
print("----------------------")
print("CPU Time:", cpu_very_large_time)
print("GPU Time:", gpu_very_large_time)
print("GPU Speedup:", round(speedup_50m, 2), "x")


50 MILLION DATASET PERFORMANCE
----------------------
CPU Time: 0.02285477100008393
GPU Time: 0.0011556350000319071
GPU Speedup: 19.78 x


GPU kernel speedup

19.78×

The GPU kernel was approximately 19.78 times faster than the CPU calculation in this run.

**Measure data-transfer time**

In [ ]:

start = time.perf_counter()

d_transfer_50m = cuda.to_device(very_large_dataset)

cuda.synchronize()

transfer_50m_time = time.perf_counter() - start

print("DATA TRANSFER RESULT")
print("----------------------")
print("Dataset size:", len(very_large_dataset))
print("Transfer time:", transfer_50m_time, "seconds")


DATA TRANSFER RESULT
----------------------
Dataset size: 50000000
Transfer time: 0.05171031600002607 seconds


**Calculate overall GPU performance**

In [ ]:

total_gpu_50m_time = transfer_50m_time + gpu_very_large_time

overall_speedup_50m = cpu_very_large_time / total_gpu_50m_time

print("50 MILLION DATASET PERFORMANCE")
print("----------------------")
print("CPU Time:", cpu_very_large_time)
print("GPU Kernel Time:", gpu_very_large_time)
print("Data Transfer Time:", transfer_50m_time)
print("Total GPU Time:", total_gpu_50m_time)
print("Overall Speedup:", round(overall_speedup_50m, 2), "x")


50 MILLION DATASET PERFORMANCE
----------------------
CPU Time: 0.02285477100008393
GPU Kernel Time: 0.0011556350000319071
Data Transfer Time: 0.05171031600002607
Total GPU Time: 0.052865951000057976
Overall Speedup: 0.43 x


50-million performance results

Measurement	Time

CPU calculation	0.02285 s

GPU calculation	0.00116 s

Data transfer	0.05171 s

Total GPU time	0.05287 s

Overall speedup: 0.43×

What does this mean?
- GPU calculation is extremely fast.
- Data transfer takes much longer than GPU calculation.
- Including transfer time, CPU is faster for this experiment.

**This is exactly why our project measures both kernel performance and end-to-end performance.**

**Create a performance comparison table**

In [ ]:

import pandas as pd

performance_data = {
    "Dataset Size": ["1 Million", "10 Million", "50 Million"],
    "CPU Time (seconds)": [
        cpu_time,
        cpu_large_time,
        cpu_very_large_time
    ],
    "GPU Kernel Time (seconds)": [
        gpu_time,
        gpu_large_time,
        gpu_very_large_time
    ],
    "GPU Transfer Time (seconds)": [
        np.nan,
        transfer_time,
        transfer_50m_time
    ]
}

performance_df = pd.DataFrame(performance_data)

print(performance_df.to_string(index=False))


Dataset Size  CPU Time (seconds)  GPU Kernel Time (seconds)  GPU Transfer Time (seconds)
   1 Million            0.000678                   0.000712                          NaN
  10 Million            0.010354                   0.005475                     0.010995
  50 Million            0.022855                   0.001156                     0.051710


What we observed

Dataset	     CPU time	       GPU kernel time	   Overall GPU time*

1 Million	   0.000678 s	        0.000712 s	       Not measured

10 Million	 0.010354 s	        0.005475 s	        0.016470 s

50 Million	0.022855 s	         0.001156 s	        0.052866 s

*Overall GPU time = transfer time + kernel time.

Important findings:

- For 10M and 50M, GPU kernel execution was faster in these measurements.
- But transferring data to the GPU took additional time.
- When transfer time is included, the GPU was slower overall than the CPU in both cases.
- These are initial measurements. We should repeat the timings before treating them as final results.

**CPU-only cell:**

In [ ]:
very_large_cpu_average = np.mean(very_large_dataset)
very_large_cpu_min = np.min(very_large_dataset)
very_large_cpu_max = np.max(very_large_dataset)

print("CPU RESULTS - 50M DATASET")
print("Average:", very_large_cpu_average)
print("Minimum:", very_large_cpu_min)
print("Maximum:", very_large_cpu_max)

CPU RESULTS - 50M DATASET
Average: 500.50745
Minimum: 1.0
Maximum: 1000.0


**Calculate GPU results**

In [ ]:
very_large_gpu_average = very_large_gpu_sum / len(very_large_dataset)

print("GPU RESULTS - 50M DATASET")
print("Average:", very_large_gpu_average)
print("Minimum:", gpu_min)
print("Maximum:", gpu_max)

GPU RESULTS - 50M DATASET
Average: 500.50745
Minimum: 1.0
Maximum: 1000.0


**GPU minimum and maximum for 50M**

In [ ]:
d_very_large = cuda.to_device(very_large_dataset)

blocks = 256
threads = 256

d_min_partials = cuda.device_array(blocks, dtype=np.float32)
d_max_partials = cuda.device_array(blocks, dtype=np.float32)

d_min_result = cuda.device_array(1, dtype=np.float32)
d_max_result = cuda.device_array(1, dtype=np.float32)

find_min[blocks, threads](d_very_large, d_min_partials, len(very_large_dataset))
final_min[1, threads](d_min_partials, d_min_result, blocks)

find_max[blocks, threads](d_very_large, d_max_partials, len(very_large_dataset))
final_max[1, threads](d_max_partials, d_max_result, blocks)

cuda.synchronize()

print("GPU Minimum:", d_min_result.copy_to_host()[0])
print("GPU Maximum:", d_max_result.copy_to_host()[0])

GPU Minimum: 1.0
GPU Maximum: 1000.0


**Verify correctness automatically**

In [ ]:
print("CPU-GPU CORRECTNESS CHECK")

print("Sum:", np.isclose(very_large_cpu_sum, very_large_gpu_sum))
print("Average:", np.isclose(very_large_cpu_average, very_large_gpu_average))
print("Minimum:", np.isclose(very_large_cpu_min, d_min_result.copy_to_host()[0]))
print("Maximum:", np.isclose(very_large_cpu_max, d_max_result.copy_to_host()[0]))

CPU-GPU CORRECTNESS CHECK
Sum: True
Average: True
Minimum: True
Maximum: True


**Measure GPU execution time**

In [ ]:
cuda.synchronize()
start = time.perf_counter()

partial_sum[blocks, threads](
    d_very_large, d_very_large_partials, len(very_large_dataset)
)
final_sum[1, threads](
    d_very_large_partials, d_very_large_result, blocks
)

find_min[blocks, threads](d_very_large, d_min_partials, len(very_large_dataset))
final_min[1, threads](d_min_partials, d_min_result, blocks)

find_max[blocks, threads](d_very_large, d_max_partials, len(very_large_dataset))
final_max[1, threads](d_max_partials, d_max_result, blocks)

cuda.synchronize()
all_operations_gpu_time = time.perf_counter() - start

print("GPU time for Sum, Minimum and Maximum:",
      all_operations_gpu_time, "seconds")

GPU time for Sum, Minimum and Maximum: 0.0028907750001962995 seconds


**CPU execution time**

In [30]:
start = time.perf_counter()

cpu_sum_check = np.sum(very_large_dataset)
cpu_min_check = np.min(very_large_dataset)
cpu_max_check = np.max(very_large_dataset)

all_operations_cpu_time = time.perf_counter() - start

print("CPU time for Sum, Minimum and Maximum:",
      all_operations_cpu_time, "seconds")

CPU time for Sum, Minimum and Maximum: 0.0724880480001957 seconds


**Verify CPU and GPU results again**

In [31]:
print("FINAL CORRECTNESS CHECK")

print("Sum:", np.isclose(cpu_sum_check, d_very_large_result.copy_to_host()[0]))
print("Minimum:", np.isclose(cpu_min_check, d_min_result.copy_to_host()[0]))
print("Maximum:", np.isclose(cpu_max_check, d_max_result.copy_to_host()[0]))

FINAL CORRECTNESS CHECK
Sum: True
Minimum: True
Maximum: True


**Repeat CPU benchmark**

In [32]:
cpu_times = []

for i in range(5):
    start = time.perf_counter()

    np.sum(very_large_dataset)
    np.min(very_large_dataset)
    np.max(very_large_dataset)

    cpu_times.append(time.perf_counter() - start)

print("CPU timings:", cpu_times)
print("Average CPU time:", np.mean(cpu_times), "seconds")

CPU timings: [0.07037700399996538, 0.06487102900018726, 0.08864394600004744, 0.10227579200000037, 0.1681467739999789]
Average CPU time: 0.09886290900003587 seconds


**Repeat GPU benchmark 5 times**

In [33]:
gpu_times = []

for i in range(5):
    cuda.synchronize()
    start = time.perf_counter()

    partial_sum[blocks, threads](
        d_very_large, d_very_large_partials, len(very_large_dataset)
    )
    final_sum[1, threads](
        d_very_large_partials, d_very_large_result, blocks
    )

    find_min[blocks, threads](d_very_large, d_min_partials, len(very_large_dataset))
    final_min[1, threads](d_min_partials, d_min_result, blocks)

    find_max[blocks, threads](d_very_large, d_max_partials, len(very_large_dataset))
    final_max[1, threads](d_max_partials, d_max_result, blocks)

    cuda.synchronize()
    gpu_times.append(time.perf_counter() - start)

print("GPU timings:", gpu_times)
print("Average GPU time:", np.mean(gpu_times), "seconds")

GPU timings: [0.0029382210000221676, 0.0025843540001915244, 0.002515663000394852, 0.00253764199987927, 0.0024963350001598883]
Average GPU time: 0.0026144430001295406 seconds


**Calculate speedup automatically**

In [34]:
cpu_avg_time = np.mean(cpu_times)
gpu_avg_time = np.mean(gpu_times)

speedup = cpu_avg_time / gpu_avg_time

print("CPU Average:", cpu_avg_time)
print("GPU Average:", gpu_avg_time)
print("GPU Speedup:", round(speedup, 2), "x")

CPU Average: 0.09886290900003587
GPU Average: 0.0026144430001295406
GPU Speedup: 37.81 x


**Measure transfer time (5 runs)**

In [35]:
transfer_times = []

for i in range(5):
    start = time.perf_counter()

    d_temp = cuda.to_device(very_large_dataset)
    cuda.synchronize()

    transfer_times.append(time.perf_counter() - start)

print("Transfer timings:", transfer_times)
print("Average transfer time:", np.mean(transfer_times), "seconds")

Transfer timings: [0.045350829000199155, 0.044474747000094794, 0.04480586699992273, 0.045403564000025654, 0.10108753299982709]
Average transfer time: 0.056224508000013884 seconds


**Calculate end-to-end time**

In [36]:
average_transfer_time = np.mean(transfer_times)
average_kernel_time = np.mean(gpu_times)

overall_gpu_time = average_transfer_time + average_kernel_time

overall_speedup = cpu_avg_time / overall_gpu_time

print("50M DATASET - COMPLETE PERFORMANCE")
print("-----------------------------------")
print("CPU Average Time:", cpu_avg_time, "seconds")
print("GPU Transfer Time:", average_transfer_time, "seconds")
print("GPU Kernel Time:", average_kernel_time, "seconds")
print("Overall GPU Time:", overall_gpu_time, "seconds")
print("Overall GPU Speedup:", round(overall_speedup, 2), "x")

50M DATASET - COMPLETE PERFORMANCE
-----------------------------------
CPU Average Time: 0.09886290900003587 seconds
GPU Transfer Time: 0.056224508000013884 seconds
GPU Kernel Time: 0.0026144430001295406 seconds
Overall GPU Time: 0.05883895100014343 seconds
Overall GPU Speedup: 1.68 x


What did we learn?

- GPU computation is very fast.
- Moving 50 million values to GPU takes much longer than computing their results.
- Even after including transfer time, GPU is approximately 1.68× faster in this benchmark.

This is a useful finding to explain during your project presentation.

Next stage: CUDA optimization

Now we'll test whether changing the number of CUDA blocks improves performance.

**Test different block counts**

Keep threads fixed at 256.

In [37]:
block_options = [128, 256, 512, 1024]
optimization_results = []

for test_blocks in block_options:
    partials = cuda.device_array(test_blocks, dtype=np.float32)
    result = cuda.device_array(1, dtype=np.float32)

    # Warm-up
    partial_sum[test_blocks, 256](
        d_very_large, partials, len(very_large_dataset)
    )
    final_sum[1, 256](partials, result, test_blocks)
    cuda.synchronize()

    timings = []

    for i in range(5):
        start = time.perf_counter()

        partial_sum[test_blocks, 256](
            d_very_large, partials, len(very_large_dataset)
        )
        final_sum[1, 256](partials, result, test_blocks)

        cuda.synchronize()
        timings.append(time.perf_counter() - start)

    optimization_results.append((test_blocks, np.mean(timings)))

print("BLOCK COUNT | AVERAGE GPU TIME")
for b, t in optimization_results:
    print(b, "|", t)

BLOCK COUNT | AVERAGE GPU TIME
128 | 0.000989789799950813
256 | 0.0008771994001108396
512 | 0.00087512260006406
1024 | 0.0008316204000948346


We can see that increasing the number of blocks improved GPU execution time in these measurements.

The 1024-block configuration was the fastest among the four configurations tested.

**Test correctness with 1024 blocks**

In [38]:
best_blocks = 1024

best_partials = cuda.device_array(best_blocks, dtype=np.float32)
best_result = cuda.device_array(1, dtype=np.float32)

partial_sum[best_blocks, 256](
    d_very_large, best_partials, len(very_large_dataset)
)

final_sum[1, 256](
    best_partials, best_result, best_blocks
)

cuda.synchronize()

optimized_sum = best_result.copy_to_host()[0]

print("Optimized GPU Sum:", optimized_sum)
print("CPU Sum:", very_large_cpu_sum)
print("Correct:", np.isclose(optimized_sum, very_large_cpu_sum))

Optimized GPU Sum: 2.5025372e+10
CPU Sum: 2.5025372e+10
Correct: True


1024-block configuration produces the correct result.
- Previous configuration (256 blocks): 0.000877 s
- Optimized configuration (1024 blocks): 0.000832 s

That's approximately a 5.2% improvement in kernel execution time compared with the 256-block configuration in this test.

Next step: Optimize minimum and maximum too

Our project supports four operations. We should check whether 1024 blocks also improve minimum and maximum.

In [39]:
best_blocks = 1024
threads = 256

min_partials_opt = cuda.device_array(best_blocks, dtype=np.float32)
max_partials_opt = cuda.device_array(best_blocks, dtype=np.float32)

min_result_opt = cuda.device_array(1, dtype=np.float32)
max_result_opt = cuda.device_array(1, dtype=np.float32)

start = time.perf_counter()

find_min[best_blocks, threads](
    d_very_large, min_partials_opt, len(very_large_dataset)
)
final_min[1, threads](min_partials_opt, min_result_opt, best_blocks)

find_max[best_blocks, threads](
    d_very_large, max_partials_opt, len(very_large_dataset)
)
final_max[1, threads](max_partials_opt, max_result_opt, best_blocks)

cuda.synchronize()

optimized_min_max_time = time.perf_counter() - start

print("Optimized Minimum:", min_result_opt.copy_to_host()[0])
print("Optimized Maximum:", max_result_opt.copy_to_host()[0])
print("Execution Time:", optimized_min_max_time, "seconds")

Optimized Minimum: 1.0
Optimized Maximum: 1000.0
Execution Time: 0.0021547210003518558 seconds


What we have completed
1. Tested 4 CUDA block configurations for sum.
2. Identified 1024 blocks as the fastest tested configuration.
3. Verified optimized sum correctness.
4. Verified optimized minimum and maximum correctness.

**Compare all four operations using 1024 blocks**

In [40]:
optimized_average = optimized_sum / len(very_large_dataset)
optimized_min = min_result_opt.copy_to_host()[0]
optimized_max = max_result_opt.copy_to_host()[0]

print("OPTIMIZED GPU RESULTS - 50M")
print("----------------------------")
print("Sum:", optimized_sum)
print("Average:", optimized_average)
print("Minimum:", optimized_min)
print("Maximum:", optimized_max)

print("\nCORRECTNESS CHECK")
print("Sum:", np.isclose(optimized_sum, very_large_cpu_sum))
print("Average:", np.isclose(optimized_average, very_large_cpu_average))
print("Minimum:", np.isclose(optimized_min, very_large_cpu_min))
print("Maximum:", np.isclose(optimized_max, very_large_cpu_max))

OPTIMIZED GPU RESULTS - 50M
----------------------------
Sum: 2.5025372e+10
Average: 500.50745
Minimum: 1.0
Maximum: 1000.0

CORRECTNESS CHECK
Sum: True
Average: True
Minimum: True
Maximum: True


*We've completed the core implementation, correctness testing, initial benchmarking and CUDA block-count tuning.*

**Compare optimized and original sum performance**

Let's calculate how much improvement we got from changing 256 blocks to 1024 blocks.

In [41]:
original_time = optimization_results[1][1]
optimized_time = optimization_results[3][1]

improvement = (
    (original_time - optimized_time) / original_time
) * 100

print("Original GPU Time:", original_time)
print("Optimized GPU Time:", optimized_time)
print("Performance Improvement:", round(improvement, 2), "%")

Original GPU Time: 0.0008771994001108396
Optimized GPU Time: 0.0008316204000948346
Performance Improvement: 5.2 %


**Benchmark all four operations across dataset sizes**

Before documentation, we need a consistent comparison for 1M, 10M and 50M datasets.

We'll start with 1 million values, using the optimized 1024-block configuration.

**Measure GPU execution time for all operations on 1M**

In [42]:
test_blocks = 1024
threads = 256

d_small = cuda.to_device(dataset)

sum_partials = cuda.device_array(test_blocks, dtype=np.float32)
min_partials = cuda.device_array(test_blocks, dtype=np.float32)
max_partials = cuda.device_array(test_blocks, dtype=np.float32)

sum_result = cuda.device_array(1, dtype=np.float32)
min_result = cuda.device_array(1, dtype=np.float32)
max_result = cuda.device_array(1, dtype=np.float32)

# Warm-up
partial_sum[test_blocks, threads](d_small, sum_partials, len(dataset))
final_sum[1, threads](sum_partials, sum_result, test_blocks)

find_min[test_blocks, threads](d_small, min_partials, len(dataset))
final_min[1, threads](min_partials, min_result, test_blocks)

find_max[test_blocks, threads](d_small, max_partials, len(dataset))
final_max[1, threads](max_partials, max_result, test_blocks)

cuda.synchronize()

# Benchmark
start = time.perf_counter()

partial_sum[test_blocks, threads](d_small, sum_partials, len(dataset))
final_sum[1, threads](sum_partials, sum_result, test_blocks)

find_min[test_blocks, threads](d_small, min_partials, len(dataset))
final_min[1, threads](min_partials, min_result, test_blocks)

find_max[test_blocks, threads](d_small, max_partials, len(dataset))
final_max[1, threads](max_partials, max_result, test_blocks)

cuda.synchronize()

small_gpu_time = time.perf_counter() - start

print("1M GPU execution time:", small_gpu_time)

1M GPU execution time: 0.0007870529998399434


**Benchmark 10 million values**

In [43]:
d_medium = cuda.to_device(large_dataset)

sum_partials_10 = cuda.device_array(1024, dtype=np.float32)
min_partials_10 = cuda.device_array(1024, dtype=np.float32)
max_partials_10 = cuda.device_array(1024, dtype=np.float32)

sum_result_10 = cuda.device_array(1, dtype=np.float32)
min_result_10 = cuda.device_array(1, dtype=np.float32)
max_result_10 = cuda.device_array(1, dtype=np.float32)

# Warm-up
partial_sum[1024, 256](d_medium, sum_partials_10, len(large_dataset))
final_sum[1, 256](sum_partials_10, sum_result_10, 1024)

find_min[1024, 256](d_medium, min_partials_10, len(large_dataset))
final_min[1, 256](min_partials_10, min_result_10, 1024)

find_max[1024, 256](d_medium, max_partials_10, len(large_dataset))
final_max[1, 256](max_partials_10, max_result_10, 1024)

cuda.synchronize()

# Benchmark
start = time.perf_counter()

partial_sum[1024, 256](d_medium, sum_partials_10, len(large_dataset))
final_sum[1, 256](sum_partials_10, sum_result_10, 1024)

find_min[1024, 256](d_medium, min_partials_10, len(large_dataset))
final_min[1, 256](min_partials_10, min_result_10, 1024)

find_max[1024, 256](d_medium, max_partials_10, len(large_dataset))
final_max[1, 256](max_partials_10, max_result_10, 1024)

cuda.synchronize()

medium_gpu_time = time.perf_counter() - start

print("10M GPU execution time:", medium_gpu_time)

10M GPU execution time: 0.0008002920003491454


**Benchmark 50 million values**

In [44]:
d_large_50 = d_very_large

sum_partials_50 = cuda.device_array(1024, dtype=np.float32)
min_partials_50 = cuda.device_array(1024, dtype=np.float32)
max_partials_50 = cuda.device_array(1024, dtype=np.float32)

sum_result_50 = cuda.device_array(1, dtype=np.float32)
min_result_50 = cuda.device_array(1, dtype=np.float32)
max_result_50 = cuda.device_array(1, dtype=np.float32)

# Warm-up
partial_sum[1024, 256](d_large_50, sum_partials_50, len(very_large_dataset))
final_sum[1, 256](sum_partials_50, sum_result_50, 1024)

find_min[1024, 256](d_large_50, min_partials_50, len(very_large_dataset))
final_min[1, 256](min_partials_50, min_result_50, 1024)

find_max[1024, 256](d_large_50, max_partials_50, len(very_large_dataset))
final_max[1, 256](max_partials_50, max_result_50, 1024)

cuda.synchronize()

# Benchmark
start = time.perf_counter()

partial_sum[1024, 256](d_large_50, sum_partials_50, len(very_large_dataset))
final_sum[1, 256](sum_partials_50, sum_result_50, 1024)

find_min[1024, 256](d_large_50, min_partials_50, len(very_large_dataset))
final_min[1, 256](min_partials_50, min_result_50, 1024)

find_max[1024, 256](d_large_50, max_partials_50, len(very_large_dataset))
final_max[1, 256](max_partials_50, max_result_50, 1024)

cuda.synchronize()

large_gpu_time = time.perf_counter() - start

print("50M GPU execution time:", large_gpu_time)

50M GPU execution time: 0.002562455999395752


In [45]:
# CPU benchmark: Sum, Minimum and Maximum
import time
import numpy as np

datasets = {
    "1 Million": dataset,
    "10 Million": large_dataset,
    "50 Million": very_large_dataset
}

cpu_benchmark_results = {}

for name, data in datasets.items():
    times = []

    # Run 5 times for reliable timing
    for _ in range(5):
        start = time.perf_counter()

        cpu_sum = np.sum(data)
        cpu_min = np.min(data)
        cpu_max = np.max(data)

        elapsed = time.perf_counter() - start
        times.append(elapsed)

    average_time = np.mean(times)
    cpu_benchmark_results[name] = average_time

    print(f"{name}: {average_time:.6f} seconds")

print("\nCPU benchmark completed.")

1 Million: 0.001107 seconds
10 Million: 0.012133 seconds
50 Million: 0.071803 seconds

CPU benchmark completed.


**Speedup = CPU execution time ÷ GPU kernel execution time. GPU timings exclude data transfer from CPU to GPU.**

What these results tell us
- 1M: GPU is slightly faster.
- 10M: GPU is significantly faster.
- 50M: GPU is about 28 times faster for kernel execution.

Benchmark comparison completed

One important point: these results compare CPU execution with GPU kernel execution only. They do not include the time needed to transfer data to the GPU.

Let's measure GPU data-transfer time for all three dataset sizes so we can calculate the actual end-to-end GPU performance.

In [46]:
# Measure GPU data transfer time (5 runs)

datasets = [
    ("1 Million", dataset),
    ("10 Million", large_dataset),
    ("50 Million", very_large_dataset)
]

transfer_results = {}

for name, data in datasets:
    times = []

    for _ in range(5):
        start = time.perf_counter()

        gpu_data = cuda.to_device(data)
        cuda.synchronize()

        end = time.perf_counter()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    transfer_results[name] = avg_time

    print(f"{name}: {avg_time:.6f} seconds")

print("\nGPU DATA TRANSFER BENCHMARK COMPLETED")

1 Million: 0.001730 seconds
10 Million: 0.009533 seconds
50 Million: 0.073074 seconds

GPU DATA TRANSFER BENCHMARK COMPLETED



# FINAL PERFORMANCE ANALYSIS

## 1. Project Overview
This project implements parallel reduction operations using CUDA and Numba on an NVIDIA Tesla T4 GPU.

The system performs four operations on large numerical datasets:
- Sum
- Average
- Minimum
- Maximum

## 2. CPU vs GPU Performance

The following table compares CPU execution time, GPU kernel execution time, and CPU-to-GPU data transfer time.

| Dataset Size | CPU Time (s) | GPU Kernel Time (s) | Transfer Time (s) |
|---|---:|---:|---:|
| 1 Million | 0.001107 | 0.000787 | 0.001730 |
| 10 Million | 0.012133 | 0.000800 | 0.009533 |
| 50 Million | 0.071803 | 0.002562 | 0.073074 |

## 3. Overall GPU Performance

Total GPU Time = GPU Kernel Time + Data Transfer Time

| Dataset Size | Total GPU Time (s) | Overall Speedup |
|---|---:|---:|
| 1 Million | 0.002517 | 0.44x |
| 10 Million | 0.010333 | 1.17x |
| 50 Million | 0.075636 | 0.95x |

## 4. CUDA Kernel Optimization

Different CUDA block configurations were tested using a 50-million-element dataset.

- Threads per block: 256
- Blocks tested: 128, 256, 512, 1024

The 1024-block configuration gave the lowest measured execution time among the tested configurations.

- 256 blocks: 0.000877 seconds
- 1024 blocks: 0.000832 seconds
- Measured improvement: approximately 5.2%

## 5. Correctness Verification

All four operations were tested by comparing GPU results with CPU results.

- Sum: Correct
- Average: Correct
- Minimum: Correct
- Maximum: Correct

The GPU results matched the CPU results for the tested dataset.

## 6. Key Observations

1. GPU kernel execution was faster than CPU execution for all three dataset sizes in these measurements.
2. Data transfer adds extra time to GPU processing.
3. GPU processing showed an overall advantage for the 10-million-element dataset.
4. For 1 million and 50 million elements, CPU execution was slightly faster when data transfer was included.
5. CUDA block configuration can affect kernel performance.

## 7. Final Conclusion

This project demonstrates how CUDA parallel programming can accelerate numerical operations using GPU processing.

It also shows that GPU performance depends not only on computation speed but also on data transfer overhead and dataset size.

The project successfully implements and verifies parallel sum, average, minimum, and maximum operations using CUDA on an NVIDIA Tesla T4 GPU.


**Small note:** These are measured timings from your current Colab runs, not guaranteed benchmark results for every system. Keep the transfer-time comparison clearly separate from kernel-only speedup in your report.